# 04 — Model Comparison
**Loads results from:** `gp_results.pkl`, `xgb_results.pkl`, `mlp_results.pkl`

**Purpose:** Compare GP, XGBoost, and MLP side-by-side so you can make an
informed decision about which model to trust for FEM input.

**Run this notebook AFTER running 01, 02, 03.**

### What to look for:

| Metric | Meaning |
|---|---|
| LOO-CV MAE | Mean absolute error on held-out points — lower is better |
| R² | Fraction of variance explained — closer to 1 is better |
| Residual pattern | Are errors random, or clustered at specific Cr%? |
| Uncertainty bands | GP: analytic. MLP: bootstrap. XGB: none (uses MAE band). |
| Behaviour between points | Does the curve make physical sense? |


---
## Cell 1 — Imports + Load All Results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, sys, os
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

sys.path.insert(0, os.path.dirname(os.path.abspath('data_utils.py')))
from data_utils import (load_data, TARGETS, PALETTE)

# ── Load both datasets for scatter points ────────────────────────────────────
DATA_PATHS = {
    'raw':       '../analysis/elastic_constants_fecr_raw.csv',
    'corrected': '../analysis/elastic_constants_fecr_corrected.csv',
}

datasets = {}
for mode, path in DATA_PATHS.items():
    d = load_data(path)
    df = d['df']
    tier = np.where(df['n_cr'] <= 9,  'A',
           np.where(df['n_cr'] <= 13, 'B', 'C'))
    datasets[mode] = dict(
        df             = df,
        X_pred         = d['X_pred'],
        x_pred_atoms   = d['x_pred_atoms'],
        targets        = d['targets'],
        flagged        = d['flagged'],
        vcr_geometry   = d['vcr_geometry'],
        cubic_enforced = d['cubic_enforced'],
        tier           = tier,
    )

# ── Load model results ────────────────────────────────────────────────────────
MODEL_FILES = {
    'GP_raw':        '../analysis/gp_results_raw.pkl',
    'GP_corrected':  '../analysis/gp_results_corrected.pkl',
    'MLP_raw':       '../analysis/mlp_results_raw.pkl',
    'MLP_corrected': '../analysis/mlp_results_corrected.pkl',
}

model_results = {}
for key, path in MODEL_FILES.items():
    with open(path, 'rb') as f:
        model_results[key] = pickle.load(f)
    print(f'Loaded: {key}')

x_pred_atoms = datasets['corrected']['x_pred_atoms']
print('\nAll results loaded.')

---
## Cell 2 — Performance Table

Side-by-side LOO-CV MAE and R² for all models and all targets.

In [ ]:
rows = []
for key, mres in model_results.items():
    model, mode = key.split('_', 1)
    for tname in TARGETS:
        if tname in mres:
            rows.append({
                'Model':         model,
                'Dataset':       mode,
                'Target':        tname,
                'LOO MAE (GPa)': round(mres[tname]['mae'], 2),
                'R²':            round(mres[tname]['r2'], 4),
            })

perf = pd.DataFrame(rows)

# Pivot tables
pivot_mae = perf.pivot_table(
    index='Target', columns=['Model','Dataset'], values='LOO MAE (GPa)')
pivot_r2  = perf.pivot_table(
    index='Target', columns=['Model','Dataset'], values='R²')

pd.set_option('display.float_format', '{:.3f}'.format)
print('LOO-CV MAE (GPa) — lower is better:')
print(pivot_mae.to_string())
print()
print('LOO-CV R² — closer to 1 is better:')
print(pivot_r2.to_string())
pd.reset_option('display.float_format')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_colours = {
    ('GP',  'raw'):       '#4878CF',
    ('GP',  'corrected'): '#2255AA',
    ('MLP', 'raw'):       '#6ACC65',
    ('MLP', 'corrected'): '#2E8B3A',
}
keys_ordered = [('GP','raw'), ('GP','corrected'),
                ('MLP','raw'), ('MLP','corrected')]
labels = ['GP raw', 'GP corr', 'MLP raw', 'MLP corr']
x  = np.arange(len(TARGETS))
bw = 0.2

for ax, pivot, title in [
    (axes[0], pivot_mae, 'LOO-CV MAE (GPa) — lower better'),
    (axes[1], pivot_r2,  'LOO-CV R²  — higher better'),
]:
    for i, (model, mode) in enumerate(keys_ordered):
        vals = []
        for tname in TARGETS:
            try:
                vals.append(pivot.loc[tname, (model, mode)])
            except KeyError:
                vals.append(np.nan)
        ax.bar(x + i*bw, vals, bw,
               label=labels[i],
               color=model_colours[(model, mode)],
               edgecolor='k', lw=0.5)
    ax.set_xticks(x + 1.5*bw)
    ax.set_xticklabels(TARGETS)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Model Performance: GP vs MLP — raw vs corrected',
             fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/model_comparison_performance.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_performance.png')

---
## Cell 3 — Prediction Overlay: All Models on One Plot

Overlay GP mean, XGBoost, and MLP predictions for each target.
Only GP uncertainty band shown (most physically meaningful).

In [ ]:
tier_specs = [
    ('A', 'black',     'o', 'Tier A'),
    ('B', 'goldenrod', 's', 'Tier B'),
    ('C', 'tomato',    'D', 'Tier C'),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for row_idx, mode in enumerate(['raw', 'corrected']):
    d    = datasets[mode]
    gp   = model_results[f'GP_{mode}']
    mlp  = model_results[f'MLP_{mode}']

    for col_idx, tname in enumerate(TARGETS):
        ax  = axes[row_idx, col_idx]
        col = PALETTE[tname]

        # GP uncertainty band
        mu_gp  = gp[tname]['mu']
        sig_gp = gp[tname].get('std', np.zeros_like(mu_gp))
        ax.fill_between(x_pred_atoms, mu_gp-sig_gp, mu_gp+sig_gp,
                        alpha=0.15, color='steelblue', label='GP ±1σ')

        # MLP bootstrap band
        mu_mlp  = mlp[tname]['mu']
        sig_mlp = mlp[tname].get('std', np.zeros_like(mu_mlp))
        ax.fill_between(x_pred_atoms, mu_mlp-sig_mlp, mu_mlp+sig_mlp,
                        alpha=0.12, color='seagreen', label='MLP ±1σ boot')

        # Prediction lines
        ax.plot(x_pred_atoms, mu_gp,  '-',  color='steelblue', lw=2.5,
                label=f'GP (R²={gp[tname]["r2"]:.3f})')
        ax.plot(x_pred_atoms, mu_mlp, '-.', color='seagreen',  lw=2.0,
                label=f'MLP (R²={mlp[tname]["r2"]:.3f})')

        # Tier markers
        for t_lbl, col_s, mk, lbl in tier_specs:
            m = ((d['tier'] == t_lbl)
                 & ~d['vcr_geometry']
                 & ~d['cubic_enforced'])
            if m.any():
                ax.scatter(d['df']['n_cr'][m],
                           d['targets'][tname][m],
                           color=col_s, s=55, marker=mk,
                           zorder=5, label=lbl)

        # Tier D
        if d['vcr_geometry'].any():
            ax.scatter(d['df']['n_cr'][d['vcr_geometry']],
                       d['targets'][tname][d['vcr_geometry']],
                       color='black', s=120, marker='x',
                       linewidths=2, zorder=6, label='Tier D')

        # Tier E
        cub = d['cubic_enforced'] & ~d['vcr_geometry']
        if cub.any():
            ax.scatter(d['df']['n_cr'][cub],
                       d['targets'][tname][cub],
                       color='purple', s=90, marker='p',
                       zorder=5, label='Tier E')

        ax.set_xlabel('Cr atoms (out of 16)')
        ax.set_ylabel(f'{tname} (GPa)')
        ax.set_title(f'{tname} [{mode}]', fontsize=10)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.set_xlim(-0.5, 16.5)

plt.suptitle('Prediction Overlay: GP vs MLP — raw vs corrected',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../analysis/model_comparison_overlay.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_overlay.png')

---
## Cell 4 — Residual Comparison Grid

9 panels: 3 models × 3 targets. Reveals which model struggles where.

In [ ]:
TIER_COL = {'A': 'steelblue', 'B': 'goldenrod', 'C': 'tomato'}

plot_order = [
    ('GP',  'raw'),
    ('GP',  'corrected'),
    ('MLP', 'raw'),
    ('MLP', 'corrected'),
]

fig, axes = plt.subplots(4, 3, figsize=(15, 14))

for row_idx, (model, mode) in enumerate(plot_order):
    d   = datasets[mode]
    res = model_results[f'{model}_{mode}']
    cols = [TIER_COL[t] for t in d['tier']]

    for col_idx, tname in enumerate(TARGETS):
        ax = axes[row_idx, col_idx]
        r  = res[tname]
        ax.bar(d['df']['n_cr'].values, r['residuals'],
               color=cols, edgecolor='k', lw=0.5)
        ax.axhline(0, color='k', lw=1)

        # vcr fail overlay
        if d['vcr_geometry'].any():
            ax.scatter(d['df']['n_cr'][d['vcr_geometry']],
                       r['residuals'][d['vcr_geometry']],
                       marker='s', s=100, facecolors='none',
                       edgecolors='black', linewidths=1.5, zorder=5)

        ax.set_title(f'{model} [{mode}] — {tname}\n'
                     f'MAE={r["mae"]:.2f} GPa  R²={r["r2"]:.3f}',
                     fontsize=9)
        ax.set_xlabel('Cr atoms')
        ax.set_ylabel('Residual (GPa)')
        ax.grid(alpha=0.3, axis='y')

from matplotlib.patches import Patch
tier_handles = [Patch(color=TIER_COL[t], label=f'Tier {t}')
                for t in ['A','B','C']]
fig.legend(handles=tier_handles, loc='upper right', fontsize=9)

plt.suptitle('LOO-CV Residuals: GP vs MLP — raw vs corrected',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../analysis/model_comparison_residuals.png', bbox_inches='tight')
plt.show()
print('Saved: model_comparison_residuals.png')

---
## Cell 5 — Model Disagreement Map

Where do the models agree? Where do they disagree?
Large disagreement at a Cr% = that region is uncertain — avoid using it
as FEM input without additional DFT data.

**Note:** Only computed where all 3 models are available.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for col_idx, mode in enumerate(['raw', 'corrected']):
    ax  = axes[col_idx]
    gp  = model_results[f'GP_{mode}']
    mlp = model_results[f'MLP_{mode}']

    for tname in TARGETS:
        spread = np.abs(gp[tname]['mu'] - mlp[tname]['mu'])
        ax.plot(x_pred_atoms, spread,
                color=PALETTE[tname], lw=2, label=tname)

    ax.axhline(5, color='tomato', lw=1, ls='--',
               label='5 GPa threshold')
    ax.set_xlabel('Cr atoms (out of 16)')
    ax.set_ylabel('|GP − MLP| (GPa)')
    ax.set_title(f'Model disagreement [{mode}]', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(-0.5, 16.5)

plt.suptitle('GP vs MLP Disagreement — raw vs corrected',
             fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/model_disagreement.png', bbox_inches='tight')
plt.show()
print('Saved: model_disagreement.png')
print('Regions above 5 GPa: unreliable for FEM — models disagree.')

---
## Cell 6 — Recommendation Summary

Printed guidance to help you choose which model output to use for FEM.

In [ ]:
print('=' * 65)
print('MODEL COMPARISON SUMMARY')
print('=' * 65)
print()
pd.set_option('display.float_format', '{:.3f}'.format)
print('LOO-CV MAE (GPa) — lower is better:')
print(pivot_mae.to_string())
print()
print('LOO-CV R² — closer to 1 is better:')
print(pivot_r2.to_string())
pd.reset_option('display.float_format')
print()

# Auto-select best model per target per constant
print('Best model per constant (by R²):')
for tname in TARGETS:
    best_key  = max(model_results.keys(),
                    key=lambda k: model_results[k][tname]['r2'])
    best_r2   = model_results[best_key][tname]['r2']
    best_mae  = model_results[best_key][tname]['mae']
    print(f'  {tname}: {best_key:<20s}  '
          f'R²={best_r2:.4f}  MAE={best_mae:.2f} GPa')

print()
print('Interpretation:')
print('  GP  — Analytic uncertainty (posterior std). Calibrated.')
print('        Recommended when uncertainty quantification matters.')
print('        Length-scale ℓ encodes physical smoothness assumption.')
print()
print('  MLP — Bootstrap uncertainty (approximate, not calibrated).')
print('        Generally lower LOO MAE on this dataset.')
print('        No physical prior — purely data-driven.')
print()
print('Recommendation for FEM:')
print('  Primary Cij: use MLP corrected (best LOO R² for C11/C12)')
print('  Uncertainty bounds: use GP corrected std (calibrated)')
print('  Flag compositions where |GP−MLP| > 5 GPa')
print('  Do NOT extrapolate beyond x_cr=0–1 with either model')
print()
print('Known caveats:')
print('  fe02cr14/15/16 — mixed magnetic setup, down-weighted not removed')
print('  Tier E tags    — cubic enforcement residual, first-order cancels')
print('  AFM init σ     — unquantified, flagged qualitatively only')
print('  N=17           — all metrics are LOO estimates on tiny dataset,')
print('                   treat R² as indicative not definitive')